In [2]:
from dotenv import load_dotenv

load_dotenv("/home/westerd/_/research_projects/tedla-hypertension/src/nlp_method/.env")


True

In [ ]:
import csv
import glob
import os
from pathlib import Path
import modin.pandas as pd

export_dir = os.getenv("EXPORT_DIR")
assert export_dir is not None
export_dir = Path(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

csv_files = glob.glob(f"{export_dir}/results__note_groups__note_details_*.csv")  # type: ignore

dfs = [pd.read_csv(f) for f in csv_files]
combined_df = pd.concat(dfs, ignore_index=True)


2025-12-04 12:39:59,841	INFO worker.py:2023 -- Started a local Ray instance.


(pid=gcs_server) [2025-12-04 12:40:28,461 E 1459310 1459310] (gcs_server) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2025-12-04 12:40:29,750 E 1459709 1459709] (raylet) main.cc:979: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(_get_index_and_columns pid=1459859) [2025-12-04 12:40:31,710 E 1459859 1460631] core_worker_process.cc:837: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
[2025-12-04 12:40:32,750 E 1458922 1459857] core_worker_process.cc:837: Failed to establish connection to the metrics export

In [10]:
term_present = combined_df[['search_term', 'window_text']].apply(
    lambda x: isinstance(x['window_text'], str) and x['search_term'] in x['window_text'],
    axis=1
)

In [36]:
combined_df = combined_df[term_present]

In [37]:
import os
import csv
from pathlib import Path
import math
import modin.pandas as mpd
from concurrent.futures import ThreadPoolExecutor

export_dir = os.getenv("EXPORT_DIR")
assert export_dir is not None
export_dir = Path(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

display("Building full output...")

rows_per_file = 1_000_000
combined_df = mpd.DataFrame(combined_df)  # Convert to Modin DataFrame if not already
total_rows = len(combined_df)
display(f'Total Rows: {total_rows}')
num_files = math.ceil(total_rows / rows_per_file)
display(f'Number of output files to create: {num_files}')

def write_chunk(i):
    display(f"Creating {i+1} of {num_files}...")
    start = i * rows_per_file
    end = min(start + rows_per_file, total_rows)
    chunk = combined_df.iloc[start:end]
    chunk.to_csv(
        export_dir / f"final_results__note_groups__note_details_{i+1}.csv",
        index=False,
        quoting=csv.QUOTE_ALL,
        quotechar='"',
        sep=","
    )

with ThreadPoolExecutor() as executor:
    executor.map(write_chunk, range(num_files))

display("Results written to CSV")

'Building full output...'

'Total Rows: 30701259'

'Number of output files to create: 31'

'Creating 1 of 31...'

'Creating 2 of 31...'

...

'Creating 29 of 31...'

'Creating 30 of 31...'

'Creating 28 of 31...'

'Creating 26 of 31...'

'Results written to CSV'